# 🚀 ResumeRocket AI

**An end-to-end, AI-powered resume tailoring pipeline.**

Upload a resume and a target job description, and ResumeRocket AI will:
- Run a structured **gap analysis** — matching skills, missing skills, and hidden strengths.
- Generate a **tailored resume**, rewritten to match the job description, with an HTML diff highlighting every addition and removal.
- Produce a matching **cover letter** in four paragraphs or fewer.
- Export the tailored resume as a **downloadable PDF**, all through a simple Gradio interface.

Built with OpenAI's structured outputs (via Pydantic), `pdfplumber` for resume parsing, and `fpdf2` for PDF generation.

---

Part of the `llm-engineering-journey` portfolio — documenting a hands-on transition from 16+ years of enterprise network engineering into AI/ML engineering.

In [180]:
import os
import tempfile
from fpdf import FPDF
import pdfplumber
from typing import Optional
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel

In [181]:
load_dotenv()

True

In [182]:
Anthropic_Base_URL = "https://api.anthropic.com/v1/"

In [183]:
anthropic_client = OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url=Anthropic_Base_URL)
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [184]:
# ---------------------------------------------------------------------------
# Pydantic models - these force the model to return exactly the fields we need
# ---------------------------------------------------------------------------
class Resumeoutput(BaseModel):
    updated_resume : str
    diff_markdown : str

class CoverLetteroutput(BaseModel):
    cover_letter : str

In [185]:
def llm_generate(
        prompt: str,
        model: str = "gpt-4o",
        temperature: float = 0.5,
        max_tokens: int = 6000,
        response_format : Optional[dict] = None,
):
    """
    Generate text using Claude Sonnet model

    This function sends a prompt to Claude and returns the generated response.
    It supports both standard text generation and structured parsing with response_format.

    Args:
        prompt (str): The prompt to send to the model, i.e.: your instructions for the AI
        model (str): The OpenAI model to use (default: "claude-sonnet-4-6")
        temperature (float): Controls randomness, where lower values make output more deterministic
        max_tokens (int): Maximum number of tokens to generate, which limits the response length
        response_format (dict): Optional format specification
        In simple terms, response_format is optional. If the user gives me a dictionary, cool!
        If they don't give me anything, just assume it's None and keep going."

    Returns:
        str or dict: The generated text or parsed structured data, depending on response_format

    Raises:
        RuntimeError: if a structured (response_format) call fails. We deliberately
        do NOT swallow this into a string the way the plain-text branch does,
        because callers like resume_generate()/generate_cover_letter() promise a
        Pydantic object back (e.g. they immediately do `result.updated_resume`).
        Silently returning an error string there causes a confusing
        "'str' object has no attribute 'updated_resume'" crash several calls
        later, instead of showing the real problem (bad/missing API key,
        no access to the model, rate limit, etc.).
    """
    if not response_format:
        try:
            response = openai_client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[{
                    "role": "system", "content" : "You are a helpful assistant specializing in resume writing and career advice."},
                    {"role" : "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error generating text: {e}"
    else:
        try:
            response = openai_client.beta.chat.completions.parse(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[{
                    "role": "system", "content" : "You are a helpful assistant specializing in resume writing and career advice."},
                    {"role" : "user", "content": prompt}],
                response_format=response_format
            )
            return response.choices[0].message.parsed
        except Exception as e:
            raise RuntimeError(f"Structured generation failed (model={model}): {e}") from e


In [186]:
# ---------------------------------------------------------------------------
# PDF -> text extraction (new, for the resume upload)
# ---------------------------------------------------------------------------

def extract_text_from_pdf(pdf_path: str)-> str:
    """
    Read every page of a PDF resume and return the combined plain text.

    Args:
        pdf_path (str): path to the uploaded PDF file on disk

    Returns:
        str: all extracted text, pages joined by a blank line
    """
    extracted_page = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                extracted_page.append(page_text)
    full_text = "\n\n".join(extracted_page)
    return full_text

In [187]:
# ---------------------------------------------------------------------------
# Gap analysis
# ---------------------------------------------------------------------------
def analyse_resume_against_job_description(
        job_description_text : str,
        resume_text: str,
        model : str = "gpt-4o",
) -> str:
    """
    Compare a resume against a job description and return a structured
    gap analysis as markdown text.

    Args:
        job_description_text (str): the target job description
        resume_text (str): the candidate's resume text
        model (str): which backend to use ("openai" is the only one wired up here)

    Returns:
        str: markdown text with four sections - requirements, matches, gaps, strengths
    """

    prompt = f"""
    Context:
    You are a career advisor and resume expert. Your task is to analyze a candidate's resume against a specific job description to assess alignment and identify areas for improvement.

    Instruction:
    Review the provided Job Description and Resume. Identify key skills, experiences, and qualifications in the Job Description and compare them to what's present in the Resume. Provide a structured analysis with the following sections:
    1. **Key Requirements from Job Description:** List the main skills, experiences, and qualifications sought by the employer.
    2. **Relevant Experience in Resume:** List the skills and experiences from the resume that match or align closely with the job requirements.
    3. **Gaps/Mismatches:** Identify important skills or qualifications from the Job Description that are missing, unclear, or underrepresented in the Resume.
    4. **Potential Strengths:** Highlight any valuable skills, experiences, or accomplishments in the resume that are not explicitly requested in the job description but could strengthen the application.

    Job Description:

    {job_description_text}

    Resume:

    {resume_text}

    Output:
    Return a clear, structured comparison with the four sections outlined above.
    """

    gap_analysis = llm_generate(
        prompt=prompt,
        temperature=0.7,
        model=model,
    )
    return gap_analysis

In [188]:
# ---------------------------------------------------------------------------
# Tailored resume generation
# ---------------------------------------------------------------------------
def resume_generate(job_description_text: str, resume_text: str, model: str ="gpt-4o", gap_analysis_text: str = None) -> Resumeoutput:

    """
    Rewrite the resume to match the job description, using the gap
    analysis as guidance, and return both the clean version and an
    HTML diff-highlighted version.

    Args:
        job_description_text (str): the target job description
        resume_text (str): the original resume text
        gap_analysis_text (str): the gap analysis produced earlier
        model (str): which backend to use ("openai" only, here)

    Returns:
        ResumeOutput: a Pydantic object with updated_resume and diff_markdown
    """

    prompt = f"""
  ### Context:
    You are an expert resume writer and editor. Your goal is to rewrite the original resume to match the target job description, using the provided tailoring suggestions and analysis.

    ---

    ### Instruction:
    1. Rewrite the entire resume to best match the Target Job Description and Gap Analysis.
    2. Improve clarity, add job-relevant keywords, and quantify achievements.
    3. Address the identified gaps directly.
    4. Keep all section headers and formatting consistent with the original resume.
    5. Return `updated_resume` as plain text, and `diff_markdown` as an HTML-highlighted
       version where additions are wrapped in a green span, e.g.
       <span style="color:green">added text</span>, and removals are wrapped in a
       red, struck-through span, e.g.
       <span style="color:red;text-decoration:line-through">removed text</span>.

    ---

    ### Input:

    **Original Resume:**

        {resume_text}



    **Target Job Description:**


        {job_description_text}



    **Gap Analysis:**


        {gap_analysis_text}
"""
    result = llm_generate(
        prompt=prompt,
        model = model,
        temperature=0.7,
        response_format=Resumeoutput)
    return result

In [189]:
# ---------------------------------------------------------------------------
# Cover letter generation
# ---------------------------------------------------------------------------

def generate_cover_letter(
        job_description_text : str,
        updated_resume : str,
        model : str = "gpt-4o",
        ) -> CoverLetteroutput:
    """
    Write a short cover letter based on the tailored resume and the
    target job description.

    Args:
        job_description_text (str): the target job description
        updated_resume_text (str): the freshly tailored resume text
        model (str): which backend to use ("openai" only, here)

    Returns:
        CoverLetterOutput: a Pydantic object with a single cover_letter field
    """
    prompt = f"""
    ### Context:
    You are a professional career coach and expert cover letter writer.

    ---

    ### Instruction:
    Write a compelling, personalized cover letter based on the Updated Resume and
    the Target Job Description. Address it generically ("Dear Hiring Manager"),
    keep it to four paragraphs or fewer, highlight key achievements, align with
    the job's responsibilities, and end with a confident, polite closing line.

    ---

    ### Input:

    **Updated Resume:**


        {updated_resume}


    **Target Job Description:**


        {job_description_text}
    """
    cover_letter = llm_generate(
        prompt=prompt,
        model=model,
        temperature=0.7,
        response_format=CoverLetteroutput
    )
    return cover_letter

In [190]:
# ---------------------------------------------------------------------------
# Resume text -> downloadable PDF
# ---------------------------------------------------------------------------

# fpdf2's built-in core fonts (Helvetica, Times, etc.) only support a strict
# Latin-1 character set. LLM output routinely contains "smart" punctuation
# (em/en dashes, curly quotes, ellipses, bullets) that falls outside that
# range and crashes PDF generation with e.g.
#   "Character '—' ... is outside the range of characters supported by
#   the font used: helvetica"
# We map the common cases to plain ASCII, then use a catch-all encode/decode
# pass to silently drop anything else that still wouldn't render, instead of
# crashing the whole pipeline over a single stray character.
SMART_CHAR_REPLACEMENTS = {
    "‘": "'", "’": "'", "‚": "'", "‛": "'",   # smart single quotes
    "“": '"', "”": '"', "„": '"', "‟": '"',   # smart double quotes
    "–": "-", "—": "-", "−": "-",                    # en dash, em dash, minus sign
    "…": "...",                                                # ellipsis
    "•": "-", "●": "-", "▪": "-",                    # bullet variants
    " ": " ",                                                 # non-breaking space
}


def sanitize_text_for_pdf(text: str) -> str:
    """
    Replace common "smart" punctuation with plain ASCII equivalents, then drop
    any remaining character the core PDF font can't render, so a stray unicode
    character never crashes PDF export.

    Args:
        text (str): raw text that may contain unicode punctuation

    Returns:
        str: text safe to pass to a core (non-unicode) fpdf2 font
    """
    for smart_char, ascii_char in SMART_CHAR_REPLACEMENTS.items():
        text = text.replace(smart_char, ascii_char)
    # Catch-all: silently drop anything else outside Latin-1 (rare, e.g. emoji)
    return text.encode("latin-1", errors="ignore").decode("latin-1")


def build_resume_pdf(resume_text: str, output_path: str) -> str:
    """
    Write plain resume text out to a simple, clean PDF file.

    Args:
        resume_text (str): the tailored resume text to render
        output_path (str): full file path where the PDF should be saved

    Returns:
        str: the same output_path, once the file has been written
    """
    pdf = FPDF(format="A4")  # standard A4 page, plenty of room for a resume
    pdf.set_auto_page_break(auto=True, margin=15)  # start a new page automatically near the bottom
    pdf.add_page()  # start with one page
    pdf.set_font("Helvetica", size=11)  # a simple, widely-available built-in font

    resume_text = sanitize_text_for_pdf(resume_text)  # strip characters the core font can't render
    resume_lines = resume_text.split("\n")  # split the resume into individual lines to render

    for line in resume_lines:  # explicit for loop, one line of text at a time
        clean_line = line.replace("**", "")  # strip any leftover markdown bold markers
        stripped_line = clean_line.strip()  # trimmed version, used just for the header check

        # Treat short, all-caps, non-empty lines as section headers and bold them
        if stripped_line != "" and stripped_line.isupper():
            pdf.set_font("Helvetica", style="B", size=12)
        else:
            pdf.set_font("Helvetica", size=11)

        # multi_cell defaults to new_x="RIGHT", which leaves the cursor sitting at
        # the right margin after each call. Since we use w=0 ("auto-fill to the
        # right margin"), the very next multi_cell call would then compute an
        # available width of page_width - right_margin - x == 0, crashing with
        # "Not enough horizontal space to render a single character" on the second
        # line of any resume. Resetting the cursor back to the left margin after
        # every line fixes it.
        pdf.multi_cell(0, 6, clean_line, new_x="LMARGIN", new_y="NEXT")

    pdf.output(output_path)  # save the finished PDF to disk
    return output_path


In [191]:
def run_resume_rocket_pipeline(pdf_file, job_description_text: str):
    """
    Full pipeline: extract resume text from the uploaded PDF, run the
    gap analysis, tailor the resume, build a downloadable PDF of it,
    and write a cover letter.

    Args:
        pdf_file: the Gradio file object for the uploaded resume PDF
        job_description_text (str): job description pasted into the textbox

    Returns:
        tuple: (gap_analysis_markdown, diff_html, updated_resume_text,
                resume_pdf_path, cover_letter_text)
    """
    # Guard clause: make sure both inputs were actually provided before calling any API
    if pdf_file is None or not job_description_text.strip():
        error_message = "Please upload a resume PDF and paste a job description before generating."
        return error_message, "", "", None, ""

    try:
        # Step 1: pull the raw text out of the uploaded resume PDF
        resume_text = extract_text_from_pdf(pdf_file.name)

        # Step 2: compare the resume against the job description
        gap_analysis_text = analyse_resume_against_job_description(job_description_text, resume_text, model="gpt-4o")

        # Step 3: rewrite the resume using the gap analysis as guidance
        resume_result = resume_generate(job_description_text, resume_text, gap_analysis_text=gap_analysis_text, model="gpt-4o")

        # Step 4: render the tailored resume as a downloadable PDF in a temp folder
        temp_dir = tempfile.mkdtemp()                                    # fresh temp folder per run
        resume_pdf_path = os.path.join(temp_dir, "tailored_resume.pdf")  # fixed file name inside it
        build_resume_pdf(resume_result.updated_resume, resume_pdf_path)  # write the PDF to disk

        # Step 5: write a cover letter that matches the tailored resume
        cover_letter_result = generate_cover_letter(job_description_text, resume_result.updated_resume, model="gpt-4o")

        return (
            gap_analysis_text,                 # shown in the Gap Analysis tab
            resume_result.diff_markdown,        # shown in the Changes (Diff) tab, rendered as HTML
            resume_result.updated_resume,       # shown in the Updated Resume tab
            resume_pdf_path,                    # wired to the download button
            cover_letter_result.cover_letter,   # shown in the Cover Letter tab
        )
    except Exception as e:
        # Surface the real failure in the UI instead of letting Gradio raise an
        # unhandled AttributeError / crash the callback.
        error_message = f"Something went wrong while generating your tailored resume: {e}"
        return error_message, "", "", None, ""


In [192]:
#---------------------------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------------------------
with gr.Blocks(title="Resume Rocket") as resume_rocket_app:
    gr.Markdown("# Resume Rocket")
    gr.Markdown("Upload your resume PDF and paste a job description to get a tailored resume, gap analysis, and cover letter.")

    with gr.Row():
        with gr.Column(scale=1):
            resume_pdf_input = gr.File(label="Upload Resume (PDF)", file_types=[".pdf"])
            job_description_input = gr.Textbox(label="Job Description", lines=15, placeholder="Paste the job description here...")
            generate_button = gr.Button("Generate Tailored Resume", variant="primary")

        with gr.Column(scale=2):
            with gr.Tab("Gap Analysis"):
                gap_analysis_output = gr.Markdown()
            with gr.Tab("Updated Resume"):
                updated_resume_output = gr.Markdown()
                resume_pdf_output = gr.File(label="Download Tailored Resume (PDF)")
            with gr.Tab("Changes (Diff)"):
                diff_output = gr.HTML()
            with gr.Tab("Cover Letter"):
                cover_letter_output = gr.Markdown()

    generate_button.click(
        fn=run_resume_rocket_pipeline,
        inputs=[resume_pdf_input, job_description_input],
        outputs=[gap_analysis_output, diff_output, updated_resume_output, resume_pdf_output, cover_letter_output],
    )


In [193]:
# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    resume_rocket_app.launch()

* Running on local URL:  http://127.0.0.1:7878
* To create a public link, set `share=True` in `launch()`.
